In [ ]:
from dbrepo.RestClient import RestClient
import pandas as pd
from dotenv import load_dotenv
import os 

load_dotenv()
password = os.getenv("DBREPO_PASS")
username = os.getenv("DBREPO_USER")
client = RestClient("https://test.dbrepo.tuwien.ac.at/", username=username, password=password)

containers = client.get_containers()
print(containers)

[ContainerBrief(id='6cfb3b8e-1792-4e46-871a-f3d103527203', name='mariadb-galera:11.3.2', image=ImageBrief(id='d79cb089-363c-488b-9717-649e44d8fcc5', name='mariadb', version='11.1.3', default=False), internal_name='mariadb_11_3_2', running=None, hash=None)]


In [ ]:
DB_ID = os.getenv("DB_ID")
df = client.get_database(DB_ID)

## Check all views

In [8]:
def get_joined_view_id(df):
    """Automate the view id lookup"""
    for t in df.views:
        if t.name == "drug_gdp_features_view":
            return t.id 
        
for t in df.views:
    print(t.name, t.id)

drug_gdp_features_view 6a6080f4-4117-4201-af05-876bf9eb05d5
ww_city_year_drug_summary 8550c148-db32-475c-8be6-d55e34782949


## Import view from API

In [9]:
db_id = DB_ID
view_id = get_joined_view_id(df)

response = client._wrapper(
    method="get",
    url=f"/api/v1/database/{db_id}/view/{view_id}"
)

print(response.status_code)
db = response.json()

200


In [10]:
print(db)

{'id': '6a6080f4-4117-4201-af05-876bf9eb05d5', 'name': 'drug_gdp_features_view', 'identifiers': [], 'query': 'select `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`daily_mean_concentration` as `daily_mean`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`metabolite_name` as `metabolite_name`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`ref_year` as `ref_year`, `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code` as `nuts_code`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` as `city_name`, `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`gdp` as `gdp` from `wastewater_data` join `city_map` on `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`city_name` join `gdp_data` on `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`nuts_code` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code`', 'owner': {'id': None, 'username': 'data_st

In [24]:
dict(db)["query"]

'select `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`daily_mean_concentration` as `daily_mean`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`metabolite_name` as `metabolite_name`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`ref_year` as `ref_year`, `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code` as `nuts_code`, `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` as `city_name`, `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`gdp` as `gdp` from `wastewater_data` join `city_map` on `dast_g20_wastewater_epidemiology_ndfx`.`wastewater_data`.`city_name` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`city_name` join `gdp_data` on `dast_g20_wastewater_epidemiology_ndfx`.`gdp_data`.`nuts_code` = `dast_g20_wastewater_epidemiology_ndfx`.`city_map`.`nuts_code`'

In [19]:
q = dict(db)["query"]

In [23]:
q.replace("dast_g20_wastewater_epidemiology_ndfx", "db").replace("wastewater_data", "ww").split("from")

['select `db`.`ww`.`daily_mean_concentration` as `daily_mean`, `db`.`ww`.`metabolite_name` as `metabolite_name`, `db`.`ww`.`ref_year` as `ref_year`, `db`.`city_map`.`nuts_code` as `nuts_code`, `db`.`ww`.`city_name` as `city_name`, `db`.`gdp_data`.`gdp` as `gdp` ',
 ' `ww` join `city_map` on `db`.`ww`.`city_name` = `db`.`city_map`.`city_name` join `gdp_data` on `db`.`gdp_data`.`nuts_code` = `db`.`city_map`.`nuts_code`']

In [11]:
response = client._wrapper(
    method="get",
    url=f"/api/v1/database/{db_id}/view/{view_id}/data",
    headers={"Accept": "application/json"}
)
print(response.status_code)
print(response.json())

200
[{'city_name': 'Purgstall', 'daily_mean': 22.03, 'gdp': 6574580000.0, 'metabolite_name': 'cocaine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 4.64, 'gdp': 6574580000.0, 'metabolite_name': 'MDMA', 'nuts_code': 'AT121', 'ref_year': 2020}, {'city_name': 'Purgstall', 'daily_mean': 13.74, 'gdp': 6574580000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 1.48, 'gdp': 6574580000.0, 'metabolite_name': 'methamphetamine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 30.6, 'gdp': 6574580000.0, 'metabolite_name': 'cannabis', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 24.39, 'gdp': 6574580000.0, 'metabolite_name': 'cocaine', 'nuts_code': 'AT121', 'ref_year': 2020}, {'city_name': 'Purgstall', 'daily_mean': 1.32, 'gdp': 6574580000.0, 'metabolite_name': 'methamphetamine', 'nuts_code': 'AT121', 'ref_year': 2020}, {

In [12]:
print(len(response.json()))

10


In [ ]:
# get_view_data(self, database_id: str, view_id: str, page: int = 0, size: int = 1000000) -> DataFrame:
#         """
#         Get data of a view in a database with the given database id and view id.

#         :param database_id: The database id.
#         :param view_id: The view id.
#         :param page: The result pagination number. Optional. Default: `0`.
#         :param size: The result pagination size. Optional. Default: `1000000`.

#         :returns: The view data, if successful.

#         """
#         url = f'/api/v1/database/{database_id}/view/{view_id}/data'
#         params = []
#         if page is not None and size is not None:
#             params.append(('page', page))
#             params.append(('size', size))
#         response = self._wrapper(method="get", url=url, params=params, headers={'Accept': 'application/json'})


response = client._wrapper(
    method="get",
    url=f"/api/v1/database/{db_id}/view/{view_id}/data?limit=50",
    headers={"Accept": "application/json"}
)

print(response.status_code)
print(response.json())
print(len(response.json()))

200
[{'city_name': 'Purgstall', 'daily_mean': 22.03, 'gdp': 6574580000.0, 'metabolite_name': 'cocaine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 4.64, 'gdp': 6574580000.0, 'metabolite_name': 'MDMA', 'nuts_code': 'AT121', 'ref_year': 2020}, {'city_name': 'Purgstall', 'daily_mean': 13.74, 'gdp': 6574580000.0, 'metabolite_name': 'amphetamine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 1.48, 'gdp': 6574580000.0, 'metabolite_name': 'methamphetamine', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 30.6, 'gdp': 6574580000.0, 'metabolite_name': 'cannabis', 'nuts_code': 'AT121', 'ref_year': 2019}, {'city_name': 'Purgstall', 'daily_mean': 24.39, 'gdp': 6574580000.0, 'metabolite_name': 'cocaine', 'nuts_code': 'AT121', 'ref_year': 2020}, {'city_name': 'Purgstall', 'daily_mean': 1.32, 'gdp': 6574580000.0, 'metabolite_name': 'methamphetamine', 'nuts_code': 'AT121', 'ref_year': 2020}, {

In [14]:
fetched_data = client.get_view_data(database_id = DB_ID,
                                        view_id=view_id)
#,page: int = 0, size: int = 1000000)

In [ ]:
fetched_data.drop_duplicates() # join did not have the ref_year condition -> incorrect view

,city_name,daily_mean,gdp,metabolite_name,nuts_code,ref_year
0,Bordeaux I,0.00,5.722960e+10,methamphetamine,FRI12,2019
1,Bordeaux I,0.00,5.722960e+10,methamphetamine,FRI12,2016
2,Amsterdam,1142.43,1.072809e+10,cocaine,NL321,2022
3,Bordeaux I,0.00,5.722960e+10,methamphetamine,FRI12,2017
4,Innsbruck,17.05,1.373652e+10,amphetamine,AT332,2018
...,...,...,...,...,...,...
37971,Kufstein,20.67,1.399246e+10,amphetamine,AT335,2023
37972,Santiago,64.90,2.014030e+10,cannabis,ES114,2018
37973,Santiago,30.41,2.172080e+10,MDMA,ES114,2022
37974,Rovaniemi,1.87,6.771740e+09,methamphetamine,FI1D7,2020


In [50]:
# def create_view(self, database_id: str, name: str, query: QueryDefinition, is_public: bool,
#                     is_schema_public: bool) -> ViewBrief:
#         """
#         Create a view in a database with the given database id.

#         :param database_id: The database id.
#         :param name: The name of the created view.
#         :param query: The query definition of the view.
#         :param is_public: The visibility of the data. If set to `True` the data will be publicly visible. Optional. Default: `True`.
#         :param is_schema_public: The visibility of the schema metadata. If set to `True` the schema metadata will be publicly visible. Optional. Default: `True`.

#         :returns: The created view, if successful.
#         """
#         database = self.get_database(database_id=database_id)
#         subset = query_to_subset(database, self.get_image(database.container.image.id), query)
#         url = f'/api/v1/database/{database_id}/view'
#         response = self._wrapper(method="post", url=url, force_auth=True,
#                                  payload=CreateView(name=name, query=subset, is_public=is_public,
#                                                     is_schema_public=is_schema_public))

# class QueryDefinition(BaseModel):
#     columns: List[str]
#     datasources: List[str]
#     joins: Optional[List[JoinDefinition]] = None
#     filters: Optional[List[FilterDefinition]] = None
#     orders: Optional[List[OrderDefinition]] = None

# class JoinDefinition(BaseModel):
#     type: JoinType
#     datasource: str
#     conditionals: List[ConditionalDefinition]

# class ConditionalDefinition(BaseModel):
#     column: str
#     foreign_column: str


from dbrepo.api.dto import QueryDefinition, JoinDefinition, ConditionalDefinition, CreateView

joins = [JoinDefinition(type="inner", datasource = "gdp_data", conditionals=[ConditionalDefinition(column = "dast_g20_wastewater_epidemiology_ndfx.drug_city_map_base_view.ref_year", 
                                                                                                   foreign_column = "dast_g20_wastewater_epidemiology_ndfx.gdp_data.ref_year"),
                                                                             ConditionalDefinition(column = "dast_g20_wastewater_epidemiology_ndfx.drug_city_map_base_view.nuts_code", 
                                                                                                   foreign_column = "dast_g20_wastewater_epidemiology_ndfx.gdp_data.nuts_code")])]
query = QueryDefinition(columns = ["gdp_data.ref_year", "gdp_data.gdp", "gdp_data.nuts_code"], datasources = ["dast_g20_wastewater_epidemiology_ndfx.gdp_data"])
query = QueryDefinition(columns = ["drug_city_map_base_view.ref_year", "drug_city_map_base_view.daily_mean", 
                                   "drug_city_map_base_view.nuts_code", "drug_city_map_base_view.metabolite_name"], datasources = ["dast_g20_wastewater_epidemiology_ndfx.drug_city_map_base_view"])
query = QueryDefinition(columns = ["dast_g20_wastewater_epidemiology_ndfx.gdp_data.ref_year", "dast_g20_wastewater_epidemiology_ndfx.gdp_data.gdp", 
                                   "dast_g20_wastewater_epidemiology_ndfx.gdp_data.nuts_code", "dast_g20_wastewater_epidemiology_ndfx.drug_city_map_base_view.ref_year", 
                                   "dast_g20_wastewater_epidemiology_ndfx.drug_city_map_base_view.daily_mean", "dast_g20_wastewater_epidemiology_ndfx.drug_city_map_base_view.nuts_code",
                                    "dast_g20_wastewater_epidemiology_ndfx.drug_city_map_base_view.metabolite_name"], 
                                   datasources = ["drug_city_map_base_view"], joins = joins)
from dbrepo.api.mapper import query_to_subset
database = client.get_database(database_id=DB_ID)
subset = query_to_subset(database, client.get_image(database.container.image.id), query)

MalformedError: Failed to map subset: column(s) are not in table.column notion: ['dast_g20_wastewater_epidemiology_ndfx.gdp_data.ref_year', 'dast_g20_wastewater_epidemiology_ndfx.gdp_data.gdp', 'dast_g20_wastewater_epidemiology_ndfx.gdp_data.nuts_code', 'dast_g20_wastewater_epidemiology_ndfx.drug_city_map_base_view.ref_year', 'dast_g20_wastewater_epidemiology_ndfx.drug_city_map_base_view.daily_mean', 'dast_g20_wastewater_epidemiology_ndfx.drug_city_map_base_view.nuts_code', 'dast_g20_wastewater_epidemiology_ndfx.drug_city_map_base_view.metabolite_name', ConditionalDefinition(column='dast_g20_wastewater_epidemiology_ndfx.drug_city_map_base_view.ref_year', foreign_column='dast_g20_wastewater_epidemiology_ndfx.gdp_data.ref_year'), ConditionalDefinition(column='dast_g20_wastewater_epidemiology_ndfx.drug_city_map_base_view.nuts_code', foreign_column='dast_g20_wastewater_epidemiology_ndfx.gdp_data.nuts_code')]

In [57]:

from dbrepo.api.dto import QueryDefinition, JoinDefinition, ConditionalDefinition, CreateView

joins = [JoinDefinition(type="inner", datasource = "gdp_data",
                        conditionals=[ConditionalDefinition(column = "drug_city_map_base_view.ref_year", 
                                                        foreign_column = "gdp_data.ref_year"),
                                    ConditionalDefinition(column = "drug_city_map_base_view.nuts_code", 
                                                        foreign_column = "gdp_data.nuts_code")])]
query = QueryDefinition(columns = ["gdp_data.ref_year", "gdp_data.gdp", "gdp_data.nuts_code"], datasources = ["gdp_data"])
query = QueryDefinition(columns = ["drug_city_map_base_view.ref_year", "drug_city_map_base_view.daily_mean", 
                                   "drug_city_map_base_view.nuts_code", "drug_city_map_base_view.metabolite_name"], 
                                   datasources = ["drug_city_map_base_view"])
query = QueryDefinition(columns = ["gdp_data.ref_year", "gdp_data.gdp", 
                                   "gdp_data.nuts_code", "drug_city_map_base_view.ref_year", 
                                   "drug_city_map_base_view.daily_mean", "drug_city_map_base_view.nuts_code",
                                    "drug_city_map_base_view.metabolite_name"], 
                                   datasources = ["drug_city_map_base_view"], joins = joins)
from dbrepo.api.mapper import query_to_subset
database = client.get_database(database_id=DB_ID)
subset = query_to_subset(database, client.get_image(database.container.image.id), query)

In [58]:
subset

Subset(columns=[SubsetColumn(id='981e7393-e469-4060-8d52-604ff8d96b87', alias=None), SubsetColumn(id='c06a9d89-1c79-4946-949d-85de8da323a6', alias=None), SubsetColumn(id='d9532bea-f0ed-4641-a8c5-4c8c0074bff7', alias=None), SubsetColumn(id='2a976b61-5c35-4b85-9d6f-665199080401', alias=None), SubsetColumn(id='cf3b9c15-0b6b-4b4b-83f5-d2545ed55e95', alias=None), SubsetColumn(id='8afb78a4-3aa5-4be7-9c54-436a4b1bd022', alias=None), SubsetColumn(id='87dc7d72-4181-47ea-aa24-ad9f795bcd1b', alias=None)], datasource_ids=['66310532-2d95-45bf-b417-967ad26c1e33'], joins=[Join(type=<JoinType.INNER: 'inner'>, datasource_id='fb47cfcc-9802-4b27-9665-e9b3b2faa756', conditionals=[Conditional(column_id='8afb78a4-3aa5-4be7-9c54-436a4b1bd022', foreign_column_id='c06a9d89-1c79-4946-949d-85de8da323a6'), Conditional(column_id='87dc7d72-4181-47ea-aa24-ad9f795bcd1b', foreign_column_id='981e7393-e469-4060-8d52-604ff8d96b87')])], filters=[], orders=[])

In [64]:
query = QueryDefinition(
    columns = ["gdp_data.ref_year", "gdp_data.gdp", "gdp_data.nuts_code", 
               "drug_city_map_base_view.ref_year", "drug_city_map_base_view.daily_mean", 
               "drug_city_map_base_view.nuts_code", "drug_city_map_base_view.metabolite_name"], 
    datasources = ["drug_city_map_base_view", "gdp_data"],  # Add gdp_data here
    joins = joins
)
database = client.get_database(database_id=DB_ID)
subset = query_to_subset(database, client.get_image(database.container.image.id), query)

In [67]:
joins = [JoinDefinition(
    type="inner", 
    datasource="gdp_data", 
    conditionals=[
        ConditionalDefinition(column="drug_city_map_base_view.ref_year", foreign_column="gdp_data.ref_year"),
        ConditionalDefinition(column="drug_city_map_base_view.nuts_code", foreign_column="gdp_data.nuts_code")
    ]
)]

query = QueryDefinition(
    columns=[
        "drug_city_map_base_view.ref_year", 
        "drug_city_map_base_view.daily_mean", 
        "drug_city_map_base_view.nuts_code", 
        "drug_city_map_base_view.metabolite_name",
        "gdp_data.gdp"  # Only include non-key columns from joined table
    ], 
    datasources=["drug_city_map_base_view"], 
    joins=joins
)
subset = query_to_subset(database, client.get_image(database.container.image.id), query)

MalformedError: Failed to map subset: column(s) not found in database 7 != 5

In [65]:
url = f'/api/v1/database/{DB_ID}/view'
response = client._wrapper(method="post", url=url, force_auth=True,
                            payload=CreateView(name="drug_and_gdp", query=subset, is_public=True,
                                            is_schema_public=True))

In [66]:
response.text

'{"status":"SERVICE_UNAVAILABLE","message":"Failed to create view: 400  on POST request for \\"http://data-service/api/v1/database/5cde660e-153a-4bff-8e41-69e87cda399d/view\\": \\"{\\"status\\":\\"BAD_REQUEST\\",\\"message\\":\\"Failed to create view: (conn=117582) Not unique table/alias: \'gdp_data\'\\",\\"code\\":\\"error.view.invalid\\"}\\"","code":"error.data.invalid"}'

## Local Join

In [70]:
def get_base_view_id(df):
    """Automate the view id lookup"""
    for t in df.views:
        if t.name == "drug_city_map_base_view":
            return t.id 

df = client.get_database(DB_ID)
view_id = get_base_view_id(df)

fetched_data = client.get_view_data(database_id = DB_ID,
                                        view_id=view_id)

In [71]:
fetched_data

,city_name,daily_mean,metabolite_name,nuts_code,ref_year
0,Stockholm (2),33.49,MDMA,SE110,2021
1,St. Gallen Hofen,76.00,amphetamine,CH055,2020
2,Ostrava,1.55,ketamine,CZ080,2024
3,Basel,23.09,MDMA,CH033,2013
4,Basel,585.35,cocaine,CH033,2018
...,...,...,...,...,...
3205,Barcelona,31.41,MDMA,ES511,2013
3206,Prague (2),79.63,cannabis,CZ010,2013
3207,Molina de Segura,0.00,methamphetamine,ES620,2015
3208,Kouvola,8.24,cocaine,FI1C4,2018


## Transform into dataset, ready to be used

## Test if results stay the same